# Git과 Docker 준비

이번 실습은 도구 설치와 실행 준비 상태를 각각 확인합니다. 3절의 Python 3.12 프로젝트 환경을 준비합니다. Git과 Docker가 아직 없으면 해당 실습을 생략하고 미준비 상태를 기록합니다. Docker가 아직 준비되지 않았으면 해당 상태를 출력하고 다른 확인을 계속합니다. 외부 명령을 호출하는 보조 문법은 주석을 읽으며 따라 합니다.

In [1]:
import sys
import shutil
import subprocess
from pathlib import Path
from tempfile import TemporaryDirectory

# which는 명령의 위치를 찾습니다. 개인 경로 대신 발견 여부만 출력합니다.
git_command = shutil.which("git")
uv_command = shutil.which("uv")
docker_command = shutil.which("docker")
print(f"Git 명령 발견: {git_command is not None}")
print(f"uv 명령 발견: {uv_command is not None}")
print(f"Docker 명령 발견: {docker_command is not None}")


Git 명령 발견: True
uv 명령 발견: True
Docker 명령 발견: True


## 1. Git 명령의 실제 응답

명령 위치를 찾는 것과 실제 실행은 구분합니다. `run`은 목록에 적은 프로그램을 실행합니다. `capture_output=True`는 결과를 변수에 저장하고, `text=True`는 글자로 읽게 합니다. Git 명령이 없거나 응답하지 않으면 상태를 기록하고 다음 도구를 확인합니다.

In [2]:
# 명령이 없으면 Git 미준비를 기록하고 다른 도구 점검을 계속합니다.
git_ready = False
if git_command is None:
    print("Git: 2절의 설치 안내를 확인하세요.")
else:
    git_version = subprocess.run(
        [git_command, "--version"], capture_output=True, text=True,
    )
    git_ready = git_version.returncode == 0
    print(f"Git 응답: {git_ready}")
    if git_ready:
        print(git_version.stdout.strip())


Git 응답: True
git version 2.50.1 (Apple Git-155)


## 2. 비어 있는 임시 저장소

실제 프로젝트 대신 새 임시 폴더에서 `git init`을 실행합니다. `cwd`는 해당 명령이 사용할 작업 폴더입니다. `git status --short`의 출력이 비어 있으면 이번 폴더에서 표시할 변경 파일이 없다는 뜻입니다. 커밋을 생성했다는 뜻은 아닙니다.

In [3]:
temporary_folder = TemporaryDirectory()
repository = Path(temporary_folder.name)

if git_ready:
    # -q는 초기화 안내를 줄이고, -c는 이번 명령에만 설정을 적용합니다.
    subprocess.run(
        [git_command, "-c", "init.defaultBranch=main", "init", "-q"],
        cwd=repository, capture_output=True, text=True, check=True,
    )
    before = subprocess.run(
        [git_command, "status", "--short"], cwd=repository,
        capture_output=True, text=True, check=True,
    )
    print(f"파일 생성 전 상태가 비어 있음: {before.stdout == ''}")
    assert before.stdout == ""
else:
    print("저장소 비교: Git 미준비로 실행하지 않음")


파일 생성 전 상태가 비어 있음: True


## 3. 파일 하나를 만든 뒤 비교

같은 저장소에 `hello.py` 파일 하나를 추가합니다. `??`는 Git이 아직 추적하지 않는 파일 표시입니다. 이 셀은 파일의 내용을 실행하는 것이 아니라 파일 생성 전후의 Git 상태를 비교합니다.

In [4]:
if git_ready:
    # 작은 파일 하나를 만들고 Git이 발견하는지 확인합니다.
    script_file = repository / "hello.py"
    script_file.write_text("price = 3000", encoding="utf-8")
    after = subprocess.run(
        [git_command, "status", "--short"], cwd=repository,
        capture_output=True, text=True, check=True,
    )
    print(after.stdout.strip())
    assert after.stdout.strip() == "?? hello.py"
else:
    print("파일 상태 비교: Git 미준비로 실행하지 않음")
# 임시 저장소만 정리하며 사용자의 프로젝트 파일은 바꾸지 않습니다.
temporary_folder.cleanup()


?? hello.py


## 4. Docker 클라이언트 확인

`docker --version`은 클라이언트 프로그램이 응답하는지 확인합니다. 아직 설치되지 않았다면 준비하지 않은 상태로 기록합니다. 명령이 있더라도 실행 결과를 별도로 확인합니다. 버전 문자열은 컴퓨터마다 달라질 수 있습니다.

In [5]:
docker_client_ready = False
if docker_command is None:
    print("Docker 클라이언트: 명령을 찾지 못함")
else:
    # 이 요청은 Docker 서버를 시작하거나 이미지를 내려받지 않습니다.
    client = subprocess.run(
        [docker_command, "--version"], capture_output=True, text=True,
    )
    docker_client_ready = client.returncode == 0
    print(f"Docker 클라이언트 응답: {docker_client_ready}")
    if docker_client_ready:
        print(client.stdout.strip())


Docker 클라이언트 응답: True
Docker version 29.4.1, build 055a478


## 5. Docker 데몬 응답 확인

Docker 서버는 요청을 처리하는 데몬(daemon)을 뜻하며, 출력의 Server 항목에 해당합니다. 클라이언트가 응답한 경우에만 데몬의 버전을 요청합니다. `--format`은 필요한 버전 항목만 선택합니다. 서버가 응답하지 않으면 원인을 숨겨 성공으로 표시하지 않고 미준비 상태를 기록합니다. 10초 안에 응답하지 않는 경우도 별도로 표시합니다.

In [6]:
docker_server_ready = False
if docker_client_ready:
    try:
        server = subprocess.run(
            [docker_command, "version", "--format", "{{.Server.Version}}"],
            capture_output=True, text=True, timeout=10,
        )
        docker_server_ready = server.returncode == 0 and bool(server.stdout.strip())
        print(f"Docker 서버 응답: {docker_server_ready}")
        if docker_server_ready:
            print(f"서버 버전: {server.stdout.strip()}")
    except subprocess.TimeoutExpired:
        print("Docker 서버: 10초 안에 응답하지 않음")
else:
    print("Docker 서버: 클라이언트 미준비로 확인하지 않음")


Docker 서버 응답: True
서버 버전: 29.4.1


## 6. 여섯 가지 준비 상태

Python 버전, uv 응답, 현재 커널의 가상환경 여부, Git, Docker 클라이언트, Docker 서버를 각각 읽습니다. 가상환경 여부는 현재 커널이 어떤 가상환경 안에 있다는 사실만 확인합니다. 원하는 프로젝트의 `.venv`인지 여부는 3절의 인터프리터·커널 선택 화면과 함께 확인합니다. 아래 결과는 도구 준비 상태이며 A/B 학습 수준을 채점하는 표가 아닙니다.

In [7]:
# uv 역시 위치 발견에 그치지 않고 버전 명령의 성공을 확인합니다.
uv_ready = False
if uv_command is not None:
    uv_result = subprocess.run([uv_command, "--version"], capture_output=True, text=True)
    uv_ready = uv_result.returncode == 0
python_ready = sys.version_info[:2] == (3, 12)
virtual_environment = sys.prefix != sys.base_prefix
print(f"Python 3.12: {python_ready}")
print(f"uv 응답: {uv_ready}")
print(f"현재 커널이 가상환경 내부: {virtual_environment}")
print(f"Git 응답: {git_ready}")
print(f"Docker 클라이언트 응답: {docker_client_ready}")
print(f"Docker 서버 응답: {docker_server_ready}")
assert python_ready, "Python 3.12 커널을 선택하세요."


Python 3.12: True
uv 응답: True
현재 커널이 가상환경 내부: True
Git 응답: True
Docker 클라이언트 응답: True
Docker 서버 응답: True


## 결과를 해석합니다

Git의 변경 상태는 파일 하나의 생성 전후를 비교한 실제 결과입니다. Docker 서버가 미준비이면 Docker Desktop을 시작한 뒤 다시 확인합니다. 서버가 응답해도 이미지 내려받기와 컨테이너 실행을 검증한 것은 아닙니다. Python·Git 실습과 Docker 준비를 나눠 기록합니다.